In [1]:
import pandas as pd
import numpy as np
from matplotlib.colors import Normalize, to_hex
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import colormaps


# Numeric (Gradient) Style

In [ ]:

def stylize_numeric_ranges(df, columns, value_range, cmap='RdYlGn'):
    """
    Stylizes numeric columns using a color gradient.
    """
    min_val, max_val = value_range
    norm = Normalize(vmin=min_val, vmax=max_val)
    if isinstance(cmap, str):
        cmap = colormaps.get_cmap(cmap)

    def colorize_numeric(val):
        rgba = cmap(norm(float(val)))
        return f'background-color: {to_hex(rgba)}'

    styled = df.style.map(colorize_numeric, subset=columns)
    return styled


# --- Create Sample Data ---
df = pd.DataFrame({
    "Value": np.linspace(-10, 10, 11),
    "Ratio": np.linspace(-1, 1, 11),
    "isValid": [True, False, True, True, False, True, False, False, True, True, False],
    "Category": ['A', 'B', 'D', 'C', 'E', 'A', 'C', 'F', 'B', 'C', 'A']
})
df['Category'] = df['Category'].astype('category')

custom_rwg = LinearSegmentedColormap.from_list(
    name='RedWhiteGreen',
    colors=['salmon', 'white', 'limegreen']
)
styled = stylize_numeric_ranges(
    df,
    columns=['Value'],
    value_range=(-10, 10),
    cmap='RdYlGn'
)
styled


,Value,Ratio,isValid,Category
0,-10.000000,-1.000000,True,A
1,-8.000000,-0.800000,False,B
2,-6.000000,-0.600000,True,D
3,-4.000000,-0.400000,True,C
4,-2.000000,-0.200000,False,E
5,0.000000,0.000000,True,A
6,2.000000,0.200000,False,C
7,4.000000,0.400000,False,F
8,6.000000,0.600000,True,B
9,8.000000,0.800000,True,C


# Numeric (Outlier) Style

In [ ]:

def stylize_numeric_outliers(
        df, columns, value_range,
        cmap=None, outlier_color='#F0A3A3'):
    min_val, max_val = value_range
    norm = Normalize(vmin=min_val, vmax=max_val)
    if isinstance(cmap, str):
        cmap = colormaps.get_cmap(cmap)

    def colorize_numeric(val):
        if pd.isna(val):
            return ''
        try:
            val = float(val)
            if val < min_val or val > max_val:
                return f'background-color: {outlier_color}'
            if cmap != None:
                rgba = cmap(norm(val))
                return f'background-color: {to_hex(rgba)}'
        except (ValueError, TypeError):
            return ''

    target_cols = columns if columns is not None else df.select_dtypes(include=np.number).columns
    return df.style.map(colorize_numeric, subset=list(target_cols))


# --- Create Sample Data ---
df = pd.DataFrame({
    "Value": np.linspace(-10, 10, 11),
    "Ratio": np.linspace(-1, 1, 11),
    "isValid": [True, False, True, True, False, True, False, False, True, True, False],
    "Category": ['A', 'B', 'D', 'C', 'E', 'A', 'C', 'F', 'B', 'C', 'A']
})
df['Category'] = df['Category'].astype('category')

custom_rwg = LinearSegmentedColormap.from_list(
    name='RedWhiteGreen',
    colors=['salmon', 'white', 'limegreen']
)
styled = stylize_numeric_outliers(
    df,
    columns=['Ratio'],
    value_range=(-.9, .5),
    # cmap='RdYlGn'
)
styled


,Value,Ratio,isValid,Category
0,-10.000000,-1.000000,True,A
1,-8.000000,-0.800000,False,B
2,-6.000000,-0.600000,True,D
3,-4.000000,-0.400000,True,C
4,-2.000000,-0.200000,False,E
5,0.000000,0.000000,True,A
6,2.000000,0.200000,False,C
7,4.000000,0.400000,False,F
8,6.000000,0.600000,True,B
9,8.000000,0.800000,True,C


# Boolean Style

In [4]:

def colorize_boolean(val):
    """Apply color styling to boolean values."""
    if isinstance(val, (bool, np.bool_)):
        return 'background-color: #A3D9B0' if val else 'background-color: #F0A3A3'
    return ''

# --- Create Sample Data ---
df = pd.DataFrame({
    "Value": np.linspace(-10, 10, 11),
    "Ratio": np.linspace(-1, 1, 11),
    "isValid": [True, False, True, True, False, True, False, False, True, True, False],
    "Category": ['A', 'B', 'D', 'C', 'E', 'A', 'C', 'F', 'B', 'C', 'A']
})
df['Category'] = df['Category'].astype('category')

custom_rwg = LinearSegmentedColormap.from_list(
    name='RedWhiteGreen',
    colors=['salmon', 'white', 'limegreen']
)

styled = df.style.map(colorize_boolean, subset=['isValid'])
styled


,Value,Ratio,isValid,Category
0,-10.000000,-1.000000,True,A
1,-8.000000,-0.800000,False,B
2,-6.000000,-0.600000,True,D
3,-4.000000,-0.400000,True,C
4,-2.000000,-0.200000,False,E
5,0.000000,0.000000,True,A
6,2.000000,0.200000,False,C
7,4.000000,0.400000,False,F
8,6.000000,0.600000,True,B
9,8.000000,0.800000,True,C


# Categorical Style

In [ ]:

def categorical_map(df, columns=None, cmap='viridis'):
    """
    Stylizes categorical columns by mapping unique values to a colormap.
    """
    cmap_func = colormaps.get_cmap(cmap)

    for col in columns:
        unique_vals = df[col].dropna().unique()
        color_map = {val: to_hex(cmap_func(i / len(unique_vals))) for i, val in enumerate(unique_vals)}
        styled = df.style.map(lambda v: f'background-color: {color_map.get(v, "")}', subset=[col])
        
    return styled


# --- Create Sample Data ---
df = pd.DataFrame({
    "Value": np.linspace(-10, 10, 11),
    "Ratio": np.linspace(-1, 1, 11),
    "isValid": [True, False, True, True, False, True, False, False, True, True, False],
    "Category": ['A', 'B', 'D', 'C', 'E', 'A', 'C', 'F', 'B', 'C', 'A']
})
df['Category'] = df['Category'].astype('category')

styled = categorical_map(df, columns=['Category'], cmap='Pastel1')
styled


,Value,Ratio,isValid,Category
0,-10.000000,-1.000000,True,A
1,-8.000000,-0.800000,False,B
2,-6.000000,-0.600000,True,D
3,-4.000000,-0.400000,True,C
4,-2.000000,-0.200000,False,E
5,0.000000,0.000000,True,A
6,2.000000,0.200000,False,C
7,4.000000,0.400000,False,F
8,6.000000,0.600000,True,B
9,8.000000,0.800000,True,C


# Using Custom Pandas Accessor

In [6]:
# Import pandas and your corrected extension module
import pandas as pd
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import styler_extensions # This registers the 'style_ext' DataFrame accessor

# --- Create Sample Data ---
df = pd.DataFrame({
    "Value": np.linspace(-10, 10, 11),
    "Ratio": np.linspace(-1, 1, 11),
    "isValid": [True, False, True, True, False, True, False, False, True, True, False],
    "Category": ['A', 'B', 'D', 'C', 'E', 'A', 'C', 'F', 'B', 'C', 'A']
})
df['Category'] = df['Category'].astype('category')
custom_rwg = LinearSegmentedColormap.from_list('RedWhiteGreen', ['salmon', 'white', 'limegreen'])

# --- Use the Fluent, Chainable Interface ---
styled_df = (df.style_ext
    .numeric_ranges(columns=['Value'], value_range=(-10, 10), cmap=custom_rwg)
    .boolean_highlights(columns=['isValid'])
    .categorical_map(columns=['Category'], cmap='Pastel1')
)
# Simply having the object as the last line in a notebook cell will display it correctly
styled_df

,Value,Ratio,isValid,Category
0,-10.000000,-1.000000,True,A
1,-8.000000,-0.800000,False,B
2,-6.000000,-0.600000,True,D
3,-4.000000,-0.400000,True,C
4,-2.000000,-0.200000,False,E
5,0.000000,0.000000,True,A
6,2.000000,0.200000,False,C
7,4.000000,0.400000,False,F
8,6.000000,0.600000,True,B
9,8.000000,0.800000,True,C
